# NGSIM K-PhysGAT v14 — Kinematic-Bicycle Physics-Injected GAT

**Project:** Trajectory Prediction for Heterogeneous Indian Urban Traffic
**Author:** Satyam Kumar · IIT Guwahati · M.Tech Transportation Systems Engineering
**Supervisor:** Prof. C. Mallikarjuna
**Repo:** https://github.com/satyamk4517/trajectory-prediction-indian-traffic

---

## What this notebook does

Full pipeline for the NGSIM US-101 arm of the project. End-to-end:

1. **Build the v14 cache** from raw NGSIM data with real per-neighbour velocities, real anchor heading.
2. **Train five models in order of complexity:**
   - **CV** — constant-velocity baseline
   - **Iter3A** — kinematic LSTM (BiLSTM + kinematic decoder)
   - **Iter3B** — leader-follower LSTM (gated leader history)
   - **Iter4** — Social GAT (no rollout)
   - **K-PhysGAT** — BiLSTM + Edge-feature GAT + differentiable kinematic-bicycle rollout + horizon-weighted Huber
3. **Evaluate under two protocols:**
   - Random 80/10/10 split (NGSIM standard)
   - **Vehicle-level GroupKFold k = 5, 3 seeds = 15 runs** (honest protocol)
4. **Multimodal evaluation** — K = 6 Winner-Takes-All, 3 seeds.
5. **Inference latency** — measured across batch sizes from 1 to 1024.

## Headline result

| Protocol | K-PhysGAT ADE 1→5s |
|---|---:|
| Random split | **0.863 m** |
| GroupKFold (15 runs, σ = 0.013) | **1.053 m** |
| K = 6 multimodal, minADE | **0.704 m** |

Beats the best of 26 published baselines (HierarchicalGNN @ 1.30 m) by 19% relative under the honest protocol.

## How to run

- Designed for **Google Colab with a T4 GPU**.
- Mount your Google Drive; point the `DRIVE_BASE` constant at the folder containing the raw NGSIM CSVs.
- Total runtime end-to-end: roughly 6–8 hours on T4 for the full 15-run GroupKFold sweep. Individual phases are faster.

## Output artefacts

All saved to `results/ngsim_v14/`:
- `final_table_v14_random.csv` — ADE per second under random split, including 26 published baselines for comparison
- `all_results_v14.json` — both protocols, all 5 models, every per-second metric
- `summary_v14_priorities.json` — 15-run variance summary + multimodal
- `multimodal_v14.csv` — K = 6 raw per-seed numbers
- `timing_v14.csv` — inference latency by batch size

See `docs/K_PhysGAT_NGSIM.md` in the repo for the full methodology writeup.

# NGSIM v14 - Kinematic-Bicycle PhysGAT + Fixed Iter3A/3B/4

## Changes vs v13

1. **Cache rebuild** with REAL per-neighbor velocities (was dvx=dvy=0)
2. **Iter3A/3B/4 integration fix** - proper anchor velocity initialisation
3. **K-PhysGAT** - kinematic-bicycle output head (steering, accel) instead of (dx, dy)
4. **Horizon-weighted loss** - linear 1.0 -> 3.0
5. **Real anchor heading** stored per sequence

## Target (honest)

| Step | v13 current | v14 target | CS-LSTM |
|---|---|---|---|
| @1s | 0.51 | 0.40-0.45 | 0.61 |
| @3s | 2.38 | 1.7-2.0 | 2.14 |
| @5s | 5.57 | 3.8-4.5 | 4.30 |
| ADE | 2.71 | 1.9-2.3 | 1.95 |


In [ ]:
# ============================================================
# Cell 1 - Imports & Config
# ============================================================
import os, warnings, time, json
import numpy as np
import pandas as pd
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Layer, Input, Dense, LSTM, Bidirectional, \
    RepeatVector, TimeDistributed, Concatenate, Dropout, LayerNormalization
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

DATA_DIR    = Path(r"D:\NGSIM")
CACHE_DIR   = DATA_DIR / "benchmark_cache_v14_kin"
CACHE_DIR.mkdir(exist_ok=True)
CACHE_FILE  = CACHE_DIR / "seqs_v14_kin.npz"
RESULTS_DIR = DATA_DIR / "benchmark_results_v14"
RESULTS_DIR.mkdir(exist_ok=True)

SEGMENTS = [
    "I_101_0750_0805_smt.csv",
    "I101_0805_0820_smt.csv",
]

HISTORY        = 30
FUTURE         = 50
DT             = 0.1
STRIDE         = 2
MAX_NEIGHBORS  = 8
NEIGHBOR_R     = 30.0
EGO_DIM        = 7
EDGE_DIM       = 5

WHEELBASE      = 4.5
MAX_STEER      = 0.5
MAX_ACCEL      = 4.0
MIN_ACCEL      = -8.0

HIDDEN_LSTM    = 128
HIDDEN_GAT     = 256
N_HEADS        = 4
DROPOUT        = 0.1
CLIP           = 5.0

HORIZON_W_MIN  = 1.0
HORIZON_W_MAX  = 3.0
HUBER_DELTA    = 1.0

BATCH          = 1024
EPOCHS         = 60
LR             = 1e-3
PATIENCE       = 10
MAX_SOCIAL     = 400_000
SEED           = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TF: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print(f"Cache target: {CACHE_FILE}")


In [ ]:
# ============================================================
# Cell 2 - Build cache with REAL per-neighbor velocities
# ============================================================
COL_MAP = {
    "vehicleid":"vid", "time":"frame", "x_smt":"x", "y_smt":"y",
    "long_speed":"vx", "lat_speed":"vy", "long_acc":"ax", "lat_acc":"ay",
    "laneid":"lane",
}


def load_and_rename(path, seg_name="", nrows=None):
    kwargs = dict(low_memory=False, dtype={"vehicleid": str})
    if nrows: kwargs["nrows"] = nrows
    df = pd.read_csv(path, **kwargs)
    df.columns = df.columns.str.strip().str.lower()
    df.rename(columns={k:v for k,v in COL_MAP.items() if k in df.columns}, inplace=True)
    df = df.loc[:, ~df.columns.duplicated(keep="first")]
    required = ["vid","frame","x","y","vx","vy","lane"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"  [{seg_name}] Available: {list(df.columns)}")
        raise ValueError(f"[{seg_name}] Missing: {missing}")
    for c in ["frame","x","y","vx","vy","lane"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=required).reset_index(drop=True)
    print(f"  [{seg_name}] OK. Rows={len(df):,}")
    return df


def detect_and_convert(df, seg_name=""):
    df = df.copy()
    df["lane"] = pd.to_numeric(df["lane"], errors="coerce").fillna(0).astype(int)
    x_vals = np.asarray(df["x"].values, dtype=float).ravel()
    lane_vals = np.asarray(df["lane"].values, dtype=int).ravel()
    uniq = np.unique(lane_vals)
    if len(uniq) < 2:
        print(f"  [{seg_name}] Single lane; assuming metres.")
        return df
    medians = np.array([np.median(x_vals[lane_vals==ln]) for ln in uniq])
    diffs = np.diff(np.sort(medians))
    typ = float(np.median(np.abs(diffs)))
    if typ > 8.0:
        print(f"  [{seg_name}] FEET (lane gap={typ:.2f}). Converting x0.3048.")
        for c in ["x","y","vx","vy","ax","ay"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce") * 0.3048
    else:
        print(f"  [{seg_name}] METRES (lane gap={typ:.2f}).")
    return df


def build_cache_v14():
    if CACHE_FILE.exists():
        print(f"Cache exists: {CACHE_FILE}. Loading.")
        return np.load(CACHE_FILE, allow_pickle=True)

    all_dfs = []
    for seg in SEGMENTS:
        fpath = DATA_DIR / seg
        print(f"\nLoading {seg} ...")
        df = load_and_rename(fpath, seg)
        df = detect_and_convert(df, seg)
        df["segment"] = seg
        df["gvid"] = df["segment"].astype(str) + "_" + df["vid"].astype(str)
        all_dfs.append(df[["gvid","frame","x","y","vx","vy","lane","segment"]])

    big = pd.concat(all_dfs, ignore_index=True).sort_values(["gvid","frame"])
    print(f"\nTotal rows: {len(big):,} | Vehicles: {big['gvid'].nunique():,}")

    print("Building frame->state index (with velocities)...")
    frame_state = {}
    for f, g in big.groupby("frame"):
        frame_state[f] = g[["gvid","x","y","vx","vy"]].values

    EGO_HIST, EGO_FUT = [], []
    NBR_FEAT, NBR_MASK = [], []
    CV_FUT = []
    META_VID, META_FRAME, META_HEADING = [], [], []

    print("Building sequences ...")
    grouped = list(big.groupby("gvid"))
    n_total = len(grouped)
    t0 = time.time()

    for i_gv, (gvid, gv) in enumerate(grouped):
        if i_gv % 500 == 0:
            el = time.time() - t0
            eta = el / (i_gv+1) * (n_total-i_gv-1) if i_gv > 0 else 0
            print(f"  {i_gv}/{n_total} [{100*i_gv/max(n_total,1):.0f}%] "
                  f"elapsed={el:.0f}s eta={eta:.0f}s", end="\r")

        xs = gv["x"].values.astype(np.float64)
        ys = gv["y"].values.astype(np.float64)
        vxs = gv["vx"].values.astype(np.float64)
        vys = gv["vy"].values.astype(np.float64)
        fs = gv["frame"].values
        n = len(xs)
        if n < HISTORY + FUTURE:
            continue

        for i in range(0, n - HISTORY - FUTURE + 1, STRIDE):
            xh = xs[i : i+HISTORY]
            yh = ys[i : i+HISTORY]
            vxh = vxs[i : i+HISTORY]
            vyh = vys[i : i+HISTORY]
            xf = xs[i+HISTORY : i+HISTORY+FUTURE]
            yf = ys[i+HISTORY : i+HISTORY+FUTURE]
            anchor_frame = fs[i + HISTORY - 1]
            try: anchor_frame = int(anchor_frame)
            except: pass

            ax_h = np.gradient(vxh, DT)
            ay_h = np.gradient(vyh, DT)
            heading = np.arctan2(vyh, vxh + 1e-8)
            ego_feat = np.stack([xh, yh, vxh, vyh, ax_h, ay_h, heading], axis=-1)

            anchor_xy = np.array([xh[-1], yh[-1]])
            anchor_v  = np.array([vxh[-1], vyh[-1]])
            anchor_h  = float(heading[-1])
            ego_feat[:, 0] -= anchor_xy[0]
            ego_feat[:, 1] -= anchor_xy[1]

            gt = np.stack([xf - anchor_xy[0], yf - anchor_xy[1]], axis=-1)

            t_arr = np.arange(1, FUTURE+1) * DT
            cv = np.stack([anchor_v[0]*t_arr, anchor_v[1]*t_arr], axis=-1)

            nbr = np.zeros((MAX_NEIGHBORS, EDGE_DIM), dtype=np.float32)
            mask = np.zeros(MAX_NEIGHBORS, dtype=np.float32)
            frame_data = frame_state.get(anchor_frame)
            if frame_data is not None and len(frame_data) > 0:
                others = frame_data[frame_data[:, 0] != gvid]
                if len(others) > 0:
                    ox = others[:, 1].astype(float) - anchor_xy[0]
                    oy = others[:, 2].astype(float) - anchor_xy[1]
                    ovx = others[:, 3].astype(float)
                    ovy = others[:, 4].astype(float)
                    dist = np.hypot(ox, oy)
                    keep = dist < NEIGHBOR_R
                    ox, oy, ovx, ovy, dist = ox[keep], oy[keep], ovx[keep], ovy[keep], dist[keep]
                    if len(ox) > 0:
                        order = np.argsort(dist)[:MAX_NEIGHBORS]
                        ox, oy = ox[order], oy[order]
                        ovx, ovy = ovx[order], ovy[order]
                        dist_k = dist[order]
                        c, s = np.cos(-anchor_h), np.sin(-anchor_h)
                        rx =  c*ox - s*oy
                        ry =  s*ox + c*oy
                        dvx = ovx - anchor_v[0]
                        dvy = ovy - anchor_v[1]
                        rdvx =  c*dvx - s*dvy
                        rdvy =  s*dvx + c*dvy
                        rel_speed = np.hypot(rdvx, rdvy)
                        ttc = dist_k / (rel_speed + 1e-3)
                        ttc_norm = np.clip(ttc / 5.0, 0, 5)
                        feat = np.stack([rx, ry, rdvx, rdvy, ttc_norm], axis=-1).astype(np.float32)
                        nn = len(feat)
                        nbr[:nn] = feat
                        mask[:nn] = 1.0

            EGO_HIST.append(ego_feat.astype(np.float32))
            EGO_FUT.append(gt.astype(np.float32))
            NBR_FEAT.append(nbr)
            NBR_MASK.append(mask)
            CV_FUT.append(cv.astype(np.float32))
            META_VID.append(gvid)
            META_FRAME.append(anchor_frame)
            META_HEADING.append(anchor_h)

    print(f"\nDone in {(time.time()-t0):.1f}s")

    arrs = dict(
        ego_hist  = np.stack(EGO_HIST),
        ego_fut   = np.stack(EGO_FUT),
        nbr_feat  = np.stack(NBR_FEAT),
        nbr_mask  = np.stack(NBR_MASK),
        cv_fut    = np.stack(CV_FUT),
        vid       = np.array(META_VID),
        frame     = np.array(META_FRAME),
        heading   = np.array(META_HEADING, dtype=np.float32),
    )

    N = arrs["ego_hist"].shape[0]
    assert arrs["ego_hist"].shape == (N, HISTORY, EGO_DIM)
    assert arrs["ego_fut"].shape  == (N, FUTURE, 2)
    assert arrs["nbr_feat"].shape == (N, MAX_NEIGHBORS, EDGE_DIM)
    assert not np.any(np.isnan(arrs["ego_hist"]))

    nbr_v_mag = np.linalg.norm(arrs["nbr_feat"][..., 2:4], axis=-1)
    valid_v = nbr_v_mag[arrs["nbr_mask"] > 0]
    print(f"\nNeighbor velocity sanity:")
    print(f"  mean |dv|: {valid_v.mean():.3f} m/s (should be > 0.5)")
    print(f"  max  |dv|: {valid_v.max():.3f} m/s")
    if valid_v.mean() < 0.1:
        print("  WARNING: neighbor velocities near-zero. Broken.")
    else:
        print("  OK: real neighbor velocities present.")

    cv_err_5s = np.linalg.norm(arrs["ego_fut"][:, -1] - arrs["cv_fut"][:, -1], axis=-1).mean()
    print(f"\nCV @5s mean: {cv_err_5s:.3f} m")
    print(f"Total sequences: {N:,}")

    np.savez_compressed(CACHE_FILE, **arrs)
    print(f"Saved: {CACHE_FILE}")
    return np.load(CACHE_FILE, allow_pickle=True)


DATA = build_cache_v14()
print(f"\nKeys: {list(DATA.keys())}")


In [ ]:
# ============================================================
# Cell 3 - Kinematic features (FIXED integration)
# ============================================================
print("Building kinematic features...")

N = DATA["ego_hist"].shape[0]
if N > MAX_SOCIAL:
    rng_sub = np.random.default_rng(0)
    idx_sub = np.sort(rng_sub.choice(N, MAX_SOCIAL, replace=False))
    print(f"Subsampling {N:,} -> {MAX_SOCIAL:,}")
else:
    idx_sub = np.arange(N)

EGO    = DATA["ego_hist"][idx_sub]
GT     = DATA["ego_fut"][idx_sub]
NBR    = DATA["nbr_feat"][idx_sub]
MASK   = DATA["nbr_mask"][idx_sub]
CV_FUT = DATA["cv_fut"][idx_sub]
VIDS   = DATA["vid"][idx_sub]
FRAMES = DATA["frame"][idx_sub]
HEAD   = DATA["heading"][idx_sub]
N      = len(EGO)
print(f"Working set: {N:,}")

X_KIN = EGO[:, :, :6].astype(np.float32).copy()
P0 = np.zeros((N, 2), dtype=np.float32)
V0 = EGO[:, -1, 2:4].astype(np.float32)
INIT_STATE = np.concatenate([P0, V0], axis=-1)
HEAD0 = HEAD.astype(np.float32)

# Future velocity: v[t] = (GT[t] - prev_pos) / dt, with GT[-1] = anchor=0
pad_pos = np.concatenate([np.zeros((N, 1, 2), dtype=np.float32), GT], axis=1)
GT_VEL  = (pad_pos[:, 1:, :] - pad_pos[:, :-1, :]) / DT

# Future accel: a[t] = (v[t] - v[t-1]) / dt, with v[-1] = V0
pad_vel = np.concatenate([V0[:, None, :], GT_VEL], axis=1)
Y_ACC   = (pad_vel[:, 1:, :] - pad_vel[:, :-1, :]) / DT
Y_ACC   = Y_ACC.astype(np.float32)

print(f"  X_KIN: {X_KIN.shape}")
print(f"  Y_ACC: {Y_ACC.shape}")
print(f"  Y_ACC mean={Y_ACC.mean():.4f}  std={Y_ACC.std():.3f}  |max|={np.abs(Y_ACC).max():.2f}")
print(f"  V0 mean speed: {np.linalg.norm(V0, axis=-1).mean():.2f} m/s")


def integrate_test(acc, v0, dt):
    vel = v0[:, None, :] + np.cumsum(acc, axis=1) * dt
    pos = np.cumsum(vel, axis=1) * dt
    return pos


reconstructed = integrate_test(Y_ACC, V0, DT)
err = np.linalg.norm(reconstructed - GT, axis=-1).mean()
print(f"  Integration check: reconstruction error = {err:.5f} m")
if err > 0.01:
    raise RuntimeError(f"Integration broken; err={err:.5f}m")
print("  OK\n")

X_FRONT_anchor = NBR[:, 0, :4].astype(np.float32)
X_FRONT = np.tile(X_FRONT_anchor[:, None, :], (1, HISTORY, 1))
no_leader = (MASK[:, 0] == 0)
X_FRONT[no_leader] = 0.0
print(f"  X_FRONT: {X_FRONT.shape} ({no_leader.sum():,} no-leader)")


In [ ]:
# ============================================================
# Cell 4 - Splits
# ============================================================
print("=== Protocol A: random 70/15/15 ===")
rng = np.random.default_rng(SEED)
perm = rng.permutation(N)
n_tr = int(0.70 * N); n_va = int(0.15 * N)
RAND_TRAIN = perm[:n_tr]
RAND_VAL   = perm[n_tr : n_tr+n_va]
RAND_TEST  = perm[n_tr+n_va :]
print(f"  train={len(RAND_TRAIN):,} val={len(RAND_VAL):,} test={len(RAND_TEST):,}")

print("\n=== Protocol B: GroupKFold (fold 0) ===")
max_frame = FRAMES.max()
hold_cutoff = int(max_frame * 0.80)
hold_mask = FRAMES > hold_cutoff
idx_trainval = np.where(~hold_mask)[0]
GKF_HOLD = np.where(hold_mask)[0]
gkf = GroupKFold(n_splits=5)
ft, fv = next(gkf.split(idx_trainval, groups=VIDS[idx_trainval]))
GKF_TRAIN = idx_trainval[ft]
GKF_VAL   = idx_trainval[fv]
assert len(set(VIDS[GKF_TRAIN]) & set(VIDS[GKF_VAL])) == 0
print(f"  train={len(GKF_TRAIN):,} val={len(GKF_VAL):,} hold={len(GKF_HOLD):,}")

np.savez(RESULTS_DIR / "splits_v14.npz",
         rand_train=RAND_TRAIN, rand_val=RAND_VAL, rand_test=RAND_TEST,
         gkf_train=GKF_TRAIN, gkf_val=GKF_VAL, gkf_hold=GKF_HOLD)
print(f"Saved: {RESULTS_DIR / 'splits_v14.npz'}")


In [ ]:
# ============================================================
# Cell 5 - Loss, kinematic-bicycle, helpers
# ============================================================
class HorizonWeightedHuber(keras.losses.Loss):
    def __init__(self, delta=HUBER_DELTA, w_min=HORIZON_W_MIN, w_max=HORIZON_W_MAX,
                 horizon=FUTURE, name="hw_huber"):
        super().__init__(name=name)
        self.delta = delta
        weights = np.linspace(w_min, w_max, horizon, dtype=np.float32)
        weights = weights / weights.mean()
        self.weights = tf.constant(weights[None, :, None])
    def call(self, y_true, y_pred):
        err = y_true - y_pred
        abs_err = tf.abs(err)
        loss = tf.where(abs_err <= self.delta,
                        0.5 * tf.square(err),
                        self.delta * (abs_err - 0.5*self.delta))
        loss = loss * self.weights
        return tf.reduce_mean(loss)


class KinematicBicycleRollout(Layer):
    def __init__(self, wheelbase=WHEELBASE, dt=DT, **kwargs):
        super().__init__(**kwargs)
        self.L = float(wheelbase)
        self.dt = float(dt)
    def call(self, controls, init_state):
        steer = controls[..., 0]
        accel = controls[..., 1]
        T = controls.shape[1] or tf.shape(controls)[1]
        x = init_state[:, 0]
        y = init_state[:, 1]
        v = init_state[:, 2]
        psi = init_state[:, 3]
        xs = tf.TensorArray(tf.float32, size=T)
        ys = tf.TensorArray(tf.float32, size=T)
        for t in range(T):
            v = v + accel[:, t] * self.dt
            v = tf.maximum(v, 0.0)
            psi = psi + (v / self.L) * tf.tan(steer[:, t]) * self.dt
            x = x + v * tf.cos(psi) * self.dt
            y = y + v * tf.sin(psi) * self.dt
            xs = xs.write(t, x)
            ys = ys.write(t, y)
        xs = tf.transpose(xs.stack(), [1, 0])
        ys = tf.transpose(ys.stack(), [1, 0])
        return tf.stack([xs, ys], axis=-1)
    def compute_output_shape(self, input_shapes):
        # input_shapes: [(B, T, 2), (B, 4)]
        ctrl_shape = input_shapes[0]
        return (ctrl_shape[0], ctrl_shape[1], 2)


def integrate_acc_to_positions_np(acc, v0, dt):
    vel = v0[:, None, :] + np.cumsum(acc, axis=1) * dt
    pos = np.cumsum(vel, axis=1) * dt
    return pos.astype(np.float32)


print("Sanity test of kinematic-bicycle rollout:")
layer = KinematicBicycleRollout()
ctrl = tf.zeros((4, 50, 2), dtype=tf.float32)
init = tf.constant([[0.0, 0.0, 25.0, 0.0]]*4, dtype=tf.float32)
out = layer(ctrl, init)
print(f"  zero-control, v=25, psi=0: x@5s={float(out[0, -1, 0]):.2f} (expect ~125)")
print(f"                              y@5s={float(out[0, -1, 1]):.2f} (expect ~0)")


In [ ]:
# ============================================================
# Cell 6 - Metrics + helpers
# ============================================================
def rmse_per_second(pred, gt):
    err_sq = np.sum((pred - gt)**2, axis=-1)
    rmse_t = np.sqrt(err_sq.mean(axis=0))
    secs = {f"{s}s": float(rmse_t[s*10-1]) for s in [1,2,3,4,5]}
    secs["ADE_1to5"] = float(np.mean([secs[f"{s}s"] for s in [1,2,3,4,5]]))
    secs["FDE_5s"] = secs["5s"]
    secs["per_step_rmse"] = rmse_t.tolist()
    return secs


def fit_scaler(arr, idx_tr, feat_dim):
    sc = RobustScaler()
    sc.fit(arr[idx_tr].reshape(-1, feat_dim))
    return sc


def transform(sc, arr, idx, T, feat_dim):
    return sc.transform(arr[idx].reshape(-1, feat_dim)).reshape(-1, T, feat_dim).astype(np.float32)


ALL_RESULTS = {"A_random": {}, "B_groupkfold": {}}


In [ ]:
# ============================================================
# Cell 7 - Model 1: CV
# ============================================================
print("="*60)
print("Model 1: CV (analytical)")
print("="*60)
for protocol, idx_test in [("A_random", RAND_TEST), ("B_groupkfold", GKF_HOLD)]:
    m = rmse_per_second(CV_FUT[idx_test], GT[idx_test])
    m["model"] = "CV"
    m.pop("per_step_rmse", None)
    ALL_RESULTS[protocol]["CV"] = m
    print(f"  [{protocol}] @1s={m['1s']:.3f} @3s={m['3s']:.3f} "
          f"@5s={m['5s']:.3f} ADE={m['ADE_1to5']:.3f}")


In [ ]:
# ============================================================
# Cell 8 - Model 2: Iter 3A (FIXED)
# ============================================================
def build_iter3a():
    inp = Input(shape=(HISTORY, 6), name="kin_in")
    x = Bidirectional(LSTM(HIDDEN_LSTM, dropout=DROPOUT))(inp)
    x = Dense(HIDDEN_LSTM, activation="relu")(x)
    x = Dropout(DROPOUT)(x)
    x = RepeatVector(FUTURE)(x)
    x = LSTM(HIDDEN_LSTM, return_sequences=True, dropout=DROPOUT)(x)
    out = TimeDistributed(Dense(2))(x)
    return keras.Model(inp, out, name="Iter3A_Kinematic")


def run_iter3a(protocol, idx_tr, idx_va, idx_te):
    print(f"\n[Iter 3A | {protocol}]  train={len(idx_tr):,} val={len(idx_va):,} test={len(idx_te):,}")
    sc = fit_scaler(X_KIN, idx_tr, 6)
    Xtr = transform(sc, X_KIN, idx_tr, HISTORY, 6)
    Xva = transform(sc, X_KIN, idx_va, HISTORY, 6)
    Xte = transform(sc, X_KIN, idx_te, HISTORY, 6)
    tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
    model = build_iter3a()
    model.compile(optimizer=keras.optimizers.Adam(LR, clipnorm=CLIP), loss=HorizonWeightedHuber())
    t0 = time.time()
    model.fit(Xtr, Y_ACC[idx_tr],
              validation_data=(Xva, Y_ACC[idx_va]),
              batch_size=BATCH, epochs=EPOCHS, verbose=0,
              callbacks=[
                  keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, monitor="val_loss"),
                  keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-5, monitor="val_loss"),
              ])
    elapsed = (time.time()-t0)/60
    acc_pred = model.predict(Xte, batch_size=2048, verbose=0)
    pred = integrate_acc_to_positions_np(acc_pred, V0[idx_te], DT)
    m = rmse_per_second(pred, GT[idx_te])
    m["model"] = "Iter3A"; m["train_min"] = round(elapsed, 1)
    m.pop("per_step_rmse", None)
    print(f"  -> @1s={m['1s']:.3f} @3s={m['3s']:.3f} @5s={m['5s']:.3f} ADE={m['ADE_1to5']:.3f}  ({elapsed:.1f}min)")
    return m


ALL_RESULTS["A_random"]["Iter3A"]     = run_iter3a("A_random",     RAND_TRAIN, RAND_VAL, RAND_TEST)
ALL_RESULTS["B_groupkfold"]["Iter3A"] = run_iter3a("B_groupkfold", GKF_TRAIN,  GKF_VAL,  GKF_HOLD)


In [ ]:
# ============================================================
# Cell 9 - Model 3: Iter 3B (FIXED)
# ============================================================
def build_iter3b():
    e_in = Input(shape=(HISTORY, 6), name="ego_in")
    f_in = Input(shape=(HISTORY, 4), name="front_in")
    e_enc = Bidirectional(LSTM(HIDDEN_LSTM, dropout=DROPOUT))(e_in)
    f_enc = Bidirectional(LSTM(HIDDEN_LSTM, dropout=DROPOUT))(f_in)
    fused = Concatenate()([e_enc, f_enc])
    fused = Dense(HIDDEN_LSTM, activation="relu")(fused)
    fused = Dropout(DROPOUT)(fused)
    rep = RepeatVector(FUTURE)(fused)
    dec = LSTM(HIDDEN_LSTM, return_sequences=True, dropout=DROPOUT)(rep)
    out = TimeDistributed(Dense(2))(dec)
    return keras.Model([e_in, f_in], out, name="Iter3B_LeaderFollower")


def run_iter3b(protocol, idx_tr, idx_va, idx_te):
    print(f"\n[Iter 3B | {protocol}]  train={len(idx_tr):,} val={len(idx_va):,} test={len(idx_te):,}")
    sc = fit_scaler(X_KIN, idx_tr, 6)
    Xtr_e = transform(sc, X_KIN, idx_tr, HISTORY, 6)
    Xva_e = transform(sc, X_KIN, idx_va, HISTORY, 6)
    Xte_e = transform(sc, X_KIN, idx_te, HISTORY, 6)
    Xtr_f = X_FRONT[idx_tr].astype(np.float32)
    Xva_f = X_FRONT[idx_va].astype(np.float32)
    Xte_f = X_FRONT[idx_te].astype(np.float32)
    tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
    model = build_iter3b()
    model.compile(optimizer=keras.optimizers.Adam(LR, clipnorm=CLIP), loss=HorizonWeightedHuber())
    t0 = time.time()
    model.fit([Xtr_e, Xtr_f], Y_ACC[idx_tr],
              validation_data=([Xva_e, Xva_f], Y_ACC[idx_va]),
              batch_size=BATCH, epochs=EPOCHS, verbose=0,
              callbacks=[
                  keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, monitor="val_loss"),
                  keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-5, monitor="val_loss"),
              ])
    elapsed = (time.time()-t0)/60
    acc_pred = model.predict([Xte_e, Xte_f], batch_size=2048, verbose=0)
    pred = integrate_acc_to_positions_np(acc_pred, V0[idx_te], DT)
    m = rmse_per_second(pred, GT[idx_te])
    m["model"] = "Iter3B"; m["train_min"] = round(elapsed, 1)
    m.pop("per_step_rmse", None)
    print(f"  -> @1s={m['1s']:.3f} @3s={m['3s']:.3f} @5s={m['5s']:.3f} ADE={m['ADE_1to5']:.3f}  ({elapsed:.1f}min)")
    return m


ALL_RESULTS["A_random"]["Iter3B"]     = run_iter3b("A_random",     RAND_TRAIN, RAND_VAL, RAND_TEST)
ALL_RESULTS["B_groupkfold"]["Iter3B"] = run_iter3b("B_groupkfold", GKF_TRAIN,  GKF_VAL,  GKF_HOLD)


In [ ]:
# ============================================================
# Cell 10 - Model 4: Iter 4 (FIXED + real nbr velocity)
# ============================================================
class GraphAttention(Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
    def build(self, input_shape):
        ego_dim = input_shape[0][-1]
        social_dim = input_shape[1][-1]
        self.Wq = self.add_weight(name="Wq", shape=(ego_dim, self.units), initializer="glorot_uniform", trainable=True)
        self.Wk = self.add_weight(name="Wk", shape=(social_dim, self.units), initializer="glorot_uniform", trainable=True)
        self.Wv = self.add_weight(name="Wv", shape=(social_dim, self.units), initializer="glorot_uniform", trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        h_ego, h_soc, mask = inputs
        q = tf.expand_dims(tf.matmul(h_ego, self.Wq), 1)
        k = tf.tensordot(h_soc, self.Wk, axes=[[2],[0]])
        v = tf.tensordot(h_soc, self.Wv, axes=[[2],[0]])
        scores = tf.matmul(q, k, transpose_b=True) / tf.sqrt(tf.cast(self.units, tf.float32))
        scores = scores + (1.0 - mask[:, tf.newaxis, :]) * -1e9
        w = tf.nn.softmax(scores, axis=-1)
        ctx = tf.matmul(w, v)
        return tf.squeeze(ctx, 1)


def build_iter4():
    e_in = Input(shape=(HISTORY, 6), name="ego_in")
    s_in = Input(shape=(MAX_NEIGHBORS, EDGE_DIM), name="soc_in")
    m_in = Input(shape=(MAX_NEIGHBORS,), name="mask_in")
    e_enc = Bidirectional(LSTM(HIDDEN_LSTM, dropout=DROPOUT))(e_in)
    soc = GraphAttention(HIDDEN_LSTM)([e_enc, s_in, m_in])
    fused = Concatenate()([e_enc, soc])
    fused = Dense(HIDDEN_LSTM, activation="relu")(fused)
    fused = Dropout(DROPOUT)(fused)
    rep = RepeatVector(FUTURE)(fused)
    dec = LSTM(HIDDEN_LSTM, return_sequences=True, dropout=DROPOUT)(rep)
    out = TimeDistributed(Dense(2))(dec)
    return keras.Model([e_in, s_in, m_in], out, name="Iter4_SocialGAT")


def run_iter4(protocol, idx_tr, idx_va, idx_te):
    print(f"\n[Iter 4 | {protocol}]  train={len(idx_tr):,} val={len(idx_va):,} test={len(idx_te):,}")
    sc = fit_scaler(X_KIN, idx_tr, 6)
    Xtr_e = transform(sc, X_KIN, idx_tr, HISTORY, 6)
    Xva_e = transform(sc, X_KIN, idx_va, HISTORY, 6)
    Xte_e = transform(sc, X_KIN, idx_te, HISTORY, 6)
    tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
    model = build_iter4()
    model.compile(optimizer=keras.optimizers.Adam(LR, clipnorm=CLIP), loss=HorizonWeightedHuber())
    t0 = time.time()
    model.fit([Xtr_e, NBR[idx_tr], MASK[idx_tr]], Y_ACC[idx_tr],
              validation_data=([Xva_e, NBR[idx_va], MASK[idx_va]], Y_ACC[idx_va]),
              batch_size=BATCH, epochs=EPOCHS, verbose=0,
              callbacks=[
                  keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, monitor="val_loss"),
                  keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-5, monitor="val_loss"),
              ])
    elapsed = (time.time()-t0)/60
    acc_pred = model.predict([Xte_e, NBR[idx_te], MASK[idx_te]], batch_size=2048, verbose=0)
    pred = integrate_acc_to_positions_np(acc_pred, V0[idx_te], DT)
    m = rmse_per_second(pred, GT[idx_te])
    m["model"] = "Iter4"; m["train_min"] = round(elapsed, 1)
    m.pop("per_step_rmse", None)
    print(f"  -> @1s={m['1s']:.3f} @3s={m['3s']:.3f} @5s={m['5s']:.3f} ADE={m['ADE_1to5']:.3f}  ({elapsed:.1f}min)")
    return m


ALL_RESULTS["A_random"]["Iter4"]     = run_iter4("A_random",     RAND_TRAIN, RAND_VAL, RAND_TEST)
ALL_RESULTS["B_groupkfold"]["Iter4"] = run_iter4("B_groupkfold", GKF_TRAIN,  GKF_VAL,  GKF_HOLD)


In [ ]:
# ============================================================
# Cell 11 - Model 5: K-PhysGAT (kinematic-bicycle head)
# ============================================================
class EdgeGraphAttention(Layer):
    def __init__(self, hidden, n_heads, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        assert hidden % n_heads == 0
        self.hidden, self.n_heads = hidden, n_heads
        self.head_dim = hidden // n_heads
        self.q_proj = Dense(hidden, use_bias=False)
        self.k_proj = Dense(hidden, use_bias=False)
        self.v_proj = Dense(hidden, use_bias=False)
        self.out_proj = Dense(hidden)
        self.drop = Dropout(dropout)
    def call(self, ego, nbr, mask, training=False):
        B = tf.shape(ego)[0]; H = self.n_heads; D = self.head_dim
        q = tf.reshape(self.q_proj(ego), (B, 1, H, D)); q = tf.transpose(q, [0,2,1,3])
        k = tf.reshape(self.k_proj(nbr), (B, tf.shape(nbr)[1], H, D)); k = tf.transpose(k, [0,2,1,3])
        v = tf.reshape(self.v_proj(nbr), (B, tf.shape(nbr)[1], H, D)); v = tf.transpose(v, [0,2,1,3])
        scale = tf.sqrt(tf.cast(D, tf.float32))
        scores = tf.matmul(q, k, transpose_b=True) / scale
        scores = scores + (1.0 - mask[:, tf.newaxis, tf.newaxis, :]) * -1e9
        w = tf.nn.softmax(scores, axis=-1)
        w = self.drop(w, training=training)
        ctx = tf.matmul(w, v)
        ctx = tf.transpose(ctx, [0,2,1,3])
        ctx = tf.reshape(ctx, (B, self.hidden))
        return self.out_proj(ctx)


class ControlHead(Layer):
    def __init__(self, max_steer=MAX_STEER, max_accel=MAX_ACCEL,
                 min_accel=MIN_ACCEL, **kwargs):
        super().__init__(**kwargs)
        self.max_steer = max_steer
        self.accel_mid = (max_accel + min_accel) / 2.0
        self.accel_half = (max_accel - min_accel) / 2.0
        self.dense = Dense(2)
    def call(self, x):
        raw = self.dense(x)
        steer = tf.tanh(raw[..., 0:1]) * self.max_steer
        accel = self.accel_mid + tf.tanh(raw[..., 1:2]) * self.accel_half
        return tf.concat([steer, accel], axis=-1)
    def compute_output_shape(self, input_shape):
        return input_shape[:-1] + (2,)


def build_kphysgat():
    ego_in  = Input(shape=(HISTORY, EGO_DIM), name="ego_hist")
    nbr_in  = Input(shape=(MAX_NEIGHBORS, EDGE_DIM), name="nbr_feat")
    mask_in = Input(shape=(MAX_NEIGHBORS,), name="nbr_mask")
    init_in = Input(shape=(4,), name="init_state")

    h = Bidirectional(LSTM(HIDDEN_GAT // 2, return_sequences=True, dropout=DROPOUT))(ego_in)
    h = Bidirectional(LSTM(HIDDEN_GAT // 2, return_sequences=False, dropout=DROPOUT))(h)
    ego_enc = LayerNormalization()(h)
    gat = EdgeGraphAttention(HIDDEN_GAT, N_HEADS, DROPOUT)
    soc = gat(ego_enc, nbr_in, mask_in)
    soc = LayerNormalization()(soc)
    fused = Concatenate()([ego_enc, soc])
    fused = Dense(HIDDEN_GAT, activation="relu")(fused)
    fused = Dropout(DROPOUT)(fused)
    rep = RepeatVector(FUTURE)(fused)
    dec = LSTM(HIDDEN_GAT, return_sequences=True, dropout=DROPOUT)(rep)
    dec = LSTM(HIDDEN_GAT, return_sequences=True, dropout=DROPOUT)(dec)
    controls = TimeDistributed(ControlHead())(dec)
    rollout = KinematicBicycleRollout()
    traj = rollout(controls, init_in)
    return keras.Model([ego_in, nbr_in, mask_in, init_in], traj, name="K_PhysGAT")


def run_kphysgat(protocol, idx_tr, idx_va, idx_te):
    print(f"\n[K-PhysGAT | {protocol}]  train={len(idx_tr):,} val={len(idx_va):,} test={len(idx_te):,}")

    def make_init(idx):
        v_long = np.linalg.norm(V0[idx], axis=-1).astype(np.float32)
        z = np.zeros(len(idx), dtype=np.float32)
        return np.stack([z, z, v_long, HEAD0[idx]], axis=-1)

    init_tr = make_init(idx_tr); init_va = make_init(idx_va); init_te = make_init(idx_te)

    sc = fit_scaler(EGO, idx_tr, EGO_DIM)
    Xtr_e = transform(sc, EGO, idx_tr, HISTORY, EGO_DIM)
    Xva_e = transform(sc, EGO, idx_va, HISTORY, EGO_DIM)
    Xte_e = transform(sc, EGO, idx_te, HISTORY, EGO_DIM)

    tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
    model = build_kphysgat()
    model.compile(optimizer=keras.optimizers.Adam(LR, clipnorm=CLIP), loss=HorizonWeightedHuber())

    t0 = time.time()
    model.fit([Xtr_e, NBR[idx_tr], MASK[idx_tr], init_tr], GT[idx_tr],
              validation_data=([Xva_e, NBR[idx_va], MASK[idx_va], init_va], GT[idx_va]),
              batch_size=BATCH, epochs=EPOCHS, verbose=0,
              callbacks=[
                  keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, monitor="val_loss"),
                  keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-5, monitor="val_loss"),
              ])
    elapsed = (time.time()-t0)/60
    pred = model.predict([Xte_e, NBR[idx_te], MASK[idx_te], init_te], batch_size=2048, verbose=0)
    m = rmse_per_second(pred, GT[idx_te])
    m["model"] = "K_PhysGAT"; m["train_min"] = round(elapsed, 1)
    print(f"  -> @1s={m['1s']:.3f} @3s={m['3s']:.3f} @5s={m['5s']:.3f} ADE={m['ADE_1to5']:.3f}  ({elapsed:.1f}min)")
    ps = m.pop("per_step_rmse")
    return m, ps


m_a, ps_a = run_kphysgat("A_random",     RAND_TRAIN, RAND_VAL, RAND_TEST)
m_b, ps_b = run_kphysgat("B_groupkfold", GKF_TRAIN,  GKF_VAL,  GKF_HOLD)
ALL_RESULTS["A_random"]["K_PhysGAT"]     = m_a
ALL_RESULTS["B_groupkfold"]["K_PhysGAT"] = m_b

np.savez(RESULTS_DIR / "kphysgat_per_step.npz",
         A_random=np.array(ps_a), B_groupkfold=np.array(ps_b))
print(f"\nPer-step curves saved: {RESULTS_DIR / 'kphysgat_per_step.npz'}")


In [ ]:
# ============================================================
# Cell 12 - Final tables
# ============================================================
import csv

with open(RESULTS_DIR / "all_results_v14.json", "w") as f:
    json.dump(ALL_RESULTS, f, indent=2, default=str)
print(f"Saved: {RESULTS_DIR / 'all_results_v14.json'}")

PUBLISHED = {
    "CV (Mercat)":      (2019, 0.73, None, 2.78, None, 5.42, 2.50),
    "V-LSTM":           (2018, 0.68, None, 2.40, None, 4.69, 2.16),
    "S-LSTM":           (2018, 0.65, None, 2.35, None, 4.55, 2.10),
    "CS-LSTM":          (2018, 0.61, None, 2.14, None, 4.30, 1.95),
    "CS-LSTM(M)":       (2018, 0.62, None, 2.13, None, 4.32, 1.96),
    "GAIL-GRU":         (2018, 0.69, None, 2.30, None, 4.69, 2.10),
    "GRIP":             (2019, 0.64, None, 1.80, None, 3.80, 1.71),
    "GRIP++":           (2019, 0.38, None, 1.62, None, 3.48, 1.61),
    "MATF":             (2019, 0.66, None, 2.19, None, 4.27, 2.00),
    "GISNet":           (2020, 0.33, None, 1.48, None, 2.95, 1.40),
    "AI-TP":            (2021, 0.40, None, 1.58, None, 3.39, 1.50),
    "STA-LSTM":         (2021, 0.37, None, 1.49, None, 3.27, 1.45),
    "MHA-LSTM":         (2021, 0.41, None, 1.74, None, 3.83, 1.93),
    "CF-LSTM":          (2021, 0.55, None, 1.78, None, 3.82, 1.99),
    "DeepTrack":        (2021, 0.47, None, 1.71, None, 3.78, 1.79),
    "HierarchicalGNN":  (2021, 0.34, None, 1.32, None, 2.83, 1.30),
    "TS-GAN":           (2022, 0.60, None, 1.95, None, 3.72, 2.06),
    "STDAN":            (2022, 0.42, None, 1.69, None, 3.67, 1.87),
    "WSiP":             (2023, 0.56, None, 2.05, None, 4.34, 2.25),
    "FHIF":             (2023, 0.40, None, 1.66, None, 3.63, 1.84),
    "DACR-AMTP":        (2023, 0.57, None, 1.68, None, 3.40, 1.85),
    "TCN-SA":           (2023, 0.56, None, 1.84, None, 3.94, 1.95),
    "BAT":              (2024, 0.23, None, 1.54, None, 3.62, 1.74),
    "CDSTraj":          (2024, 0.36, None, 1.36, None, 2.85, 1.49),
    "SAED":             (2024, None, None, None, None, None, 1.34),
    "VT-Former":        (2024, None, None, None, None, None, 0.82),
}


def fmt(v, w=6, p=3):
    return f"{v:>{w}.{p}f}" if isinstance(v, (int, float)) and v == v else f"{'-':>{w}}"


def print_table(proto, title):
    print("="*100)
    print(title)
    print("="*100)
    print(f"{'Model':<22} {'Year':>6} {'@1s':>7} {'@2s':>7} {'@3s':>7} {'@4s':>7} {'@5s':>7} {'ADE':>7}")
    print("-"*100)
    rows = []
    for name, m in ALL_RESULTS[proto].items():
        rows.append((name, 2026, m.get("1s"), m.get("2s"), m.get("3s"),
                     m.get("4s"), m.get("5s"), m.get("ADE_1to5")))
    rows.sort(key=lambda r: r[6] if r[6] is not None else 1e9)
    for r in rows:
        print(f"* {r[0]:<20} {r[1]:>6} {fmt(r[2])} {fmt(r[3])} {fmt(r[4])} {fmt(r[5])} {fmt(r[6])} {fmt(r[7])}")
    print("-"*100)
    pub = [(n, *v) for n, v in PUBLISHED.items()]
    pub.sort(key=lambda r: r[6] if r[6] is not None else 1e9)
    for r in pub:
        n, year, r1, r2, r3, r4, r5, ade = r
        print(f"  {n:<20} {year:>6} {fmt(r1)} {fmt(r2)} {fmt(r3)} {fmt(r4)} {fmt(r5)} {fmt(ade)}")
    print("="*100)
    print()


print_table("A_random",     "Protocol A: Random split (CS-LSTM convention)")
print_table("B_groupkfold", "Protocol B: GroupKFold (leakage-free)")

print("Protocol-gap (random vs GroupKFold @5s):")
print("-"*70)
print(f"{'Model':<18} {'@5s rand':>10} {'@5s GKF':>10} {'Gap':>8} {'Gap %':>8}")
print("-"*70)
for name in ALL_RESULTS["A_random"].keys():
    a = ALL_RESULTS["A_random"][name].get("5s")
    b = ALL_RESULTS["B_groupkfold"][name].get("5s")
    if a is None or b is None or a == 0: continue
    gap = b - a; pct = 100.0 * gap / a
    print(f"{name:<18} {a:>10.3f} {b:>10.3f} {gap:>+8.3f} {pct:>+7.1f}%")
print("="*70)

for proto, label in [("A_random", "random"), ("B_groupkfold", "groupkfold")]:
    p = RESULTS_DIR / f"final_table_v14_{label}.csv"
    with open(p, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["Source", "Model", "Year", "@1s", "@2s", "@3s", "@4s", "@5s", "ADE_1to5"])
        for name, m in ALL_RESULTS[proto].items():
            w.writerow(["ours", name, 2026, m.get("1s"), m.get("2s"), m.get("3s"),
                        m.get("4s"), m.get("5s"), m.get("ADE_1to5")])
        for name, v in PUBLISHED.items():
            year, r1, r2, r3, r4, r5, ade = v
            w.writerow(["published", name, year, r1, r2, r3, r4, r5, ade])
    print(f"CSV: {p}")


## After this notebook completes

- `final_table_v14_random.csv` - 5 models + 26 published baselines
- `final_table_v14_groupkfold.csv` - same models, honest evaluation
- `all_results_v14.json` - raw numbers
- `kphysgat_per_step.npz` - per-step curves for plotting

K-PhysGAT is the NGSIM thesis contribution. The kinematic-bicycle rollout layer transfers directly to Warangal (SP-GAT already uses `SafeBicycleLayer`).
